# Lab 02 - PDF, DOCX and REST API Data Extraction

**Name:** Zain Ul Wahaj
**Roll Number:** FA23-BAI-058
**Course:** Natural Language Processing

---

This notebook contains the complete solution to all **9 tasks** of Lab 02.
Each part is solved in its own code cell, directly below the question it answers,
with its output shown underneath.

**Input files** (in the `data/` folder of this repository): `sample_nlp.pdf` (8 pages, page 4 is blank on purpose),
`report_a.docx` (headings, body text, bullets, blank paragraphs and two tables) and `report_b.docx`
(a second document that overlaps with report A). **API:** https://jsonplaceholder.typicode.com

**About the two `input()` prompts.** Task 01 (d) and Task 02 (c) ask the user to type something. In the saved
outputs below the answers were `language` and `machine learning`. Run the cell again to type your own.

### Contents

1. **Task 01** - Multi-Page PDF Extraction with Word Frequency Analysis
2. **Task 02** - Structured Page Metadata Extraction and CSV Export
3. **Task 03** - PDF Text Preprocessing Pipeline for NLP
4. **Task 04** - DOCX Paragraph Classification and Statistical Analysis
5. **Task 05** - DOCX Table Extraction and Pandas Integration
6. **Task 06** - Multi-Document Comparison, Deduplication, and Frequency Report
7. **Task 07** - Multi-Endpoint API Data Joining and Analysis
8. **Task 08** - To-Do Completion Rate Analysis Across Users
9. **Task 09** - Nested JSON Flattening, DataFrame Operations, and Comment Analysis

In [2]:
# Zain Ul Wahaj | FA23-BAI-058
# Natural Language Processing - Lab 02 - PDF, DOCX and REST API Data Extraction

NAME = "Zain Ul Wahaj"
ROLL_NUMBER = "FA23-BAI-058"

print("Name        :", NAME)
print("Roll Number :", ROLL_NUMBER)
print("Course      :", "Natural Language Processing")
print("Lab         :", "Lab 02 - PDF, DOCX and REST API Data Extraction")

import re
import csv
import sys
import json
import math
import pprint
import collections

import nltk
import pandas as pd
import pdfplumber
import requests
import docx                                  # python-docx
from docx.text.paragraph import Paragraph

# stop-word list from NLTK (downloaded only if it is missing)
try:
    nltk.data.find("corpora/stopwords")
except LookupError:
    nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

PDF_PATH = "data/sample_nlp.pdf"
DOCX_A = "data/report_a.docx"
DOCX_B = "data/report_b.docx"
STOP_WORDS = set(stopwords.words("english"))

print("\nNumber of NLTK stop words available:", len(STOP_WORDS))

Name        : Zain Ul Wahaj
Roll Number : FA23-BAI-058
Course      : Natural Language Processing
Lab         : Lab 02 - PDF, DOCX and REST API Data Extraction



Number of NLTK stop words available: 198


---

## Task 01: Multi-Page PDF Extraction with Word Frequency Analysis

**Topic:** Reading a PDF page by page and counting words

### Task 01 (a)

> Open a PDF file using PyPDF2 or pdfplumber and extract text from each page individually. Store the result as a list of strings where each index represents one page. Print the total number of pages extracted.

In [3]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 01 (a)
with pdfplumber.open(PDF_PATH) as pdf:
    pages = [(page.extract_text() or "") for page in pdf.pages]      # one string per page

print("Total number of pages extracted:", len(pages))
for i, text in enumerate(pages, 1):
    print(f"   page {i}: {len(text.split()):>4} words   {text[:60].replace(chr(10), ' ')!r}")

Total number of pages extracted: 8
   page 1:  188 words   '1. Introduction to Natural Language Processing Natural langu'
   page 2:  171 words   '2. A Short History of NLP The first serious attempts at mach'
   page 3:  187 words   '3. Text Preprocessing Raw text is messy. It contains capital'
   page 4:    0 words   ''
   page 5:  184 words   '4. Language Models A language model assigns a probability to'
   page 6:  149 words   '5. Applications of NLP Search engines use NLP to understand '
   page 7:  167 words   '6. Challenges and Ethics Human language is full of ambiguity'
   page 8:   86 words   '7. Conclusion Natural language processing turns the written '


### Task 01 (b)

> Merge all page strings into a single string. Use re.sub() to remove all punctuation and digits, convert to lowercase, and split on whitespace to produce a token list. Print the total token count.

In [4]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 01 (b)
merged = " ".join(pages)

# Every character that is not a letter or whitespace (punctuation, digits, symbols) becomes a space.
# A space is used instead of an empty string so that 'https://www.nltk.org' becomes separate words
# instead of one long glued word.
letters_only = re.sub(r"[^A-Za-z\s]", " ", merged)
tokens = letters_only.lower().split()

print("Characters in the merged text :", len(merged))
print("Total token count             :", len(tokens))
print("First 15 tokens               :", tokens[:15])

Characters in the merged text : 7017
Total token count             : 1130
First 15 tokens               : ['introduction', 'to', 'natural', 'language', 'processing', 'natural', 'language', 'processing', 'usually', 'shortened', 'to', 'nlp', 'is', 'the', 'field']


### Task 01 (c)

> Build a word frequency dictionary from the token list. Remove at least 20 common stopwords. Print the top 10 most frequent words along with their counts in descending order.

In [5]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 01 (c)
print("Stop words removed:", len(STOP_WORDS), "(NLTK English list, more than the 20 required)")

word_freq = {}
for token in tokens:
    if token not in STOP_WORDS:
        word_freq[token] = word_freq.get(token, 0) + 1

print("Distinct words left:", len(word_freq))
print("\nTop 10 most frequent words:")
for rank, (word, count) in enumerate(sorted(word_freq.items(), key=lambda kv: (-kv[1], kv[0]))[:10], 1):
    print(f"   {rank:>2}. {word:<12} {count}")

Stop words removed: 198 (NLTK English list, more than the 20 required)
Distinct words left: 439

Top 10 most frequent words:
    1. language     16
    2. text         15
    3. words        12
    4. model        10
    5. models       10
    6. nlp          8
    7. word         7
    8. data         6
    9. every        6
   10. one          6


### Task 01 (d)

> Accept a keyword from the user via input(). Search through the page-wise list, split each page's text into sentences, and print every sentence containing the keyword along with its page number. If no match is found, print an appropriate message.

In [6]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 01 (d)
def search_keyword(keyword):
    """Print every sentence that contains the keyword, with its page number."""
    found = 0
    for page_number, text in enumerate(pages, 1):
        flat = " ".join(text.split())                              # line breaks inside a sentence -> spaces
        for sentence in re.split(r"(?<=[.!?])\s+", flat):          # split after . ! or ?
            if re.search(r"\b" + re.escape(keyword) + r"\b", sentence, flags=re.IGNORECASE):
                found += 1
                print(f"   [page {page_number}] {sentence}")
    if found == 0:
        print(f"   No sentence contains the keyword '{keyword}'.")
    else:
        print(f"\n   {found} sentence(s) found.")


keyword = input("Enter a keyword to search for: ").strip()
print(f"\nSentences containing '{keyword}':")
search_keyword(keyword)

print("\nAnd a keyword that is not in the document (shows the 'no match' message):")
search_keyword("zebra")

Enter a keyword to search for: language

Sentences containing 'language':
   [page 1] Introduction to Natural Language Processing Natural language processing, usually shortened to NLP, is the field of computer science that teaches machines to read, understand and produce human language.
   [page 1] Each of these stages can go wrong in its own way, which is why a careful student of language technology learns to inspect the data at every step.
   [page 2] Language turned out to be far more ambiguous and irregular than early systems assumed.
   [page 2] Hidden Markov models and n-gram language models became standard tools.
   [page 5] Language Models A language model assigns a probability to a sequence of words.
   [page 5] The oldest practical language models are n-gram models.
   [page 5] Modern language models are neural networks.
   [page 5] Perplexity is the standard way to compare language models: the lower the perplexity, the less surprised the model is by the test text.
   [page 6

---

## Task 02: Structured Page Metadata Extraction and CSV Export

**Topic:** Collecting page information into dictionaries and saving a CSV file

### Task 02 (a)

> Open a multi-page PDF (minimum 5 pages). For each page, build a dictionary with exactly these keys: page_number, raw_text, word_count, and char_count (excluding spaces). Collect all dictionaries into a list called pages_data and print the first 2 entries.

In [7]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 02 (a)
pages_data = []
with pdfplumber.open(PDF_PATH) as pdf:
    print("Pages in the PDF:", len(pdf.pages), "(minimum 5 required)\n")
    for number, page in enumerate(pdf.pages, 1):
        raw = page.extract_text() or ""
        pages_data.append({
            "page_number": number,
            "raw_text": raw,
            "word_count": len(raw.split()),
            "char_count": len(re.sub(r"\s", "", raw)),       # spaces and line breaks are not counted
        })

print("pages_data has", len(pages_data), "entries. The first 2 entries:\n")
pprint.pprint(pages_data[:2], width=110)

Pages in the PDF: 8 (minimum 5 required)



pages_data has 8 entries. The first 2 entries:

[{'char_count': 984,
  'page_number': 1,
  'raw_text': '1. Introduction to Natural Language Processing\n'
              'Natural language processing, usually shortened to NLP, is the field of computer science\n'
              'that teaches machines to read, understand and produce human language. Every day\n'
              'people write emails, send messages, post reviews and search the web, and nearly all of\n'
              'this data is unstructured text. NLP gives us the tools to turn that text into something a\n'
              'program can count, compare and learn from.\n'
              'The field sits between linguistics, computer science and machine learning. Linguistics\n'
              'explains how words, sentences and meaning are built. Computer science provides the\n'
              'data structures and algorithms to process millions of documents. Machine learning\n'
              'supplies the statistical methods that let a sys

### Task 02 (b)

> From pages_data, identify and print: (i) the page with the highest word count and its full content, (ii) the page with the lowest non-zero word count, and (iii) the average word count across all pages rounded to 2 decimal places. Handle pages that return zero words.

In [8]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 02 (b)
blank_pages = [p["page_number"] for p in pages_data if p["word_count"] == 0]
non_empty = [p for p in pages_data if p["word_count"] > 0]
print("Pages that returned zero words:", blank_pages or "none")

if non_empty:
    highest = max(non_empty, key=lambda p: p["word_count"])
    lowest = min(non_empty, key=lambda p: p["word_count"])           # zero-word pages are skipped here

    print(f"\n(i) Highest word count: page {highest['page_number']} with {highest['word_count']} words")
    print("    Full content:")
    for line in highest["raw_text"].splitlines():
        print("      ", line)

    print(f"\n(ii) Lowest non-zero word count: page {lowest['page_number']} with {lowest['word_count']} words")
else:
    print("Every page is empty, so there is no highest or lowest page.")

average_all = sum(p["word_count"] for p in pages_data) / len(pages_data) if pages_data else 0
average_non_empty = sum(p["word_count"] for p in non_empty) / len(non_empty) if non_empty else 0
print(f"\n(iii) Average word count over all {len(pages_data)} pages: {average_all:.2f}")
print(f"      (average over the {len(non_empty)} non-empty pages only: {average_non_empty:.2f})")

Pages that returned zero words: [4]

(i) Highest word count: page 1 with 188 words
    Full content:
       1. Introduction to Natural Language Processing
       Natural language processing, usually shortened to NLP, is the field of computer science
       that teaches machines to read, understand and produce human language. Every day
       people write emails, send messages, post reviews and search the web, and nearly all of
       this data is unstructured text. NLP gives us the tools to turn that text into something a
       program can count, compare and learn from.
       The field sits between linguistics, computer science and machine learning. Linguistics
       explains how words, sentences and meaning are built. Computer science provides the
       data structures and algorithms to process millions of documents. Machine learning
       supplies the statistical methods that let a system improve from examples instead of
       following only hand written rules.
       A typical

### Task 02 (c)

> Accept a search phrase via input(). Perform a case-insensitive search across all pages. For every page containing the phrase, print its page number and how many times the phrase appears. If no page contains the phrase, raise a custom exception named PhraseNotFoundError and catch it gracefully.

In [9]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 02 (c)
class PhraseNotFoundError(Exception):
    """Raised when a search phrase does not appear on any page."""


def search_phrase(phrase):
    """Return [(page_number, count), ...] for the pages that contain the phrase (case-insensitive)."""
    wanted = " ".join(phrase.lower().split())
    hits = []
    for page in pages_data:
        text = " ".join(page["raw_text"].lower().split())       # one space between words, even across line breaks
        count = text.count(wanted)
        if count:
            hits.append((page["page_number"], count))
    if not hits:
        raise PhraseNotFoundError(f"The phrase '{phrase}' does not appear on any page.")
    return hits


def run_search(phrase):
    try:
        for page_number, count in search_phrase(phrase):
            print(f"   page {page_number}: {count} time(s)")
    except PhraseNotFoundError as error:
        print("   Caught PhraseNotFoundError ->", error)


phrase = input("Enter a search phrase: ").strip()
print(f"\nSearching for '{phrase}':")
run_search(phrase)

print("\nA phrase that is not in the document (shows the exception being raised and caught):")
run_search("quantum computing")

Enter a search phrase: machine learning

Searching for 'machine learning':
   page 1: 2 time(s)

A phrase that is not in the document (shows the exception being raised and caught):
   Caught PhraseNotFoundError -> The phrase 'quantum computing' does not appear on any page.


### Task 02 (d)

> Using Python's built-in csv module (not pandas), write a file named pdf_summary.csv with columns: page_number, word_count, char_count. Then re-read that file and print its contents row by row to confirm the write was successful.

In [10]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 02 (d)
with open("pdf_summary.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["page_number", "word_count", "char_count"])
    writer.writeheader()
    for page in pages_data:
        writer.writerow({key: page[key] for key in ("page_number", "word_count", "char_count")})
print("Wrote pdf_summary.csv\n")

print("Reading it back, row by row:")
with open("pdf_summary.csv", newline="", encoding="utf-8") as f:
    rows = list(csv.reader(f))
for row in rows:
    print("   ", row)

assert len(rows) == len(pages_data) + 1          # header + one row per page
print(f"\nWrite confirmed: {len(rows) - 1} data rows plus the header row.")

Wrote pdf_summary.csv

Reading it back, row by row:
    ['page_number', 'word_count', 'char_count']
    ['1', '188', '984']
    ['2', '171', '898']
    ['3', '187', '981']
    ['4', '0', '0']
    ['5', '184', '899']
    ['6', '149', '790']
    ['7', '167', '870']
    ['8', '86', '463']

Write confirmed: 8 data rows plus the header row.


---

## Task 03: PDF Text Preprocessing Pipeline for NLP

**Topic:** Cleaning PDF text step by step so it is ready for NLP

### Task 03 (a)

> Open any PDF file and extract all text into one string. Apply the following cleaning steps in order using re.sub(): (i) remove all URLs, (ii) remove all digits, (iii) remove all non-alphabetic characters except spaces and full stops, (iv) collapse multiple consecutive spaces into a single space. Print the character count before and after cleaning.

In [11]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 03 (a)
with pdfplumber.open(PDF_PATH) as pdf:
    raw_text = " ".join((page.extract_text() or "") for page in pdf.pages)

text = raw_text
text = re.sub(r"(?:https?://|www\.)\S+", "", text)       # (i)   remove URLs
text = re.sub(r"\d+", "", text)                          # (ii)  remove digits
text = re.sub(r"[^A-Za-z. ]", " ", text)                 # (iii) keep only letters, spaces and full stops
#                                                          (line breaks and other symbols become a space so that
#                                                           two words are never glued together)
text = re.sub(r" {2,}", " ", text)                       # (iv)  collapse repeated spaces
cleaned = text.strip()

print("Characters before cleaning:", len(raw_text))
print("Characters after cleaning :", len(cleaned))
print("Removed                   :", len(raw_text) - len(cleaned))
print("\nFirst 250 characters after cleaning:")
print(cleaned[:250])

Characters before cleaning: 7017
Characters after cleaning : 6868
Removed                   : 149

First 250 characters after cleaning:
. Introduction to Natural Language Processing Natural language processing usually shortened to NLP is the field of computer science that teaches machines to read understand and produce human language. Every day people write emails send messages post 


### Task 03 (b)

> Convert the cleaned text to lowercase. Split into a list of sentences using re.split(r'[.!?]', text). Strip each sentence of leading and trailing whitespace. Remove empty strings. Print the sentence count before and after removing empty strings.

In [12]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 03 (b)
lowered = cleaned.lower()
pieces = re.split(r"[.!?]", lowered)
stripped = [piece.strip() for piece in pieces]
sentences = [piece for piece in stripped if piece]               # drop empty strings

print("Sentence count before removing empty strings:", len(stripped))
print("Sentence count after removing empty strings :", len(sentences))
print("Empty strings removed                        :", len(stripped) - len(sentences))
print("\nFirst two sentences:")
for s in sentences[:2]:
    print("  -", s)

Sentence count before removing empty strings: 75
Sentence count after removing empty strings : 67
Empty strings removed                        : 8

First two sentences:
  - introduction to natural language processing natural language processing usually shortened to nlp is the field of computer science that teaches machines to read understand and produce human language
  - every day people write emails send messages post reviews and search the web and nearly all of this data is unstructured text


### Task 03 (c)

> Tokenize each sentence into words. Remove English stopwords using nltk.corpus.stopwords.words('english'). Discard any sentence that has fewer than 5 tokens remaining after stopword removal. Print the sentence count before and after this filter and display 3 sample sentences.

In [13]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 03 (c)
english_stop = set(stopwords.words("english"))

kept = []                         # (sentence, tokens without stop words)
for sentence in sentences:
    tokens_in_sentence = re.findall(r"[a-z]+", sentence)
    content_tokens = [t for t in tokens_in_sentence if t not in english_stop]
    if len(content_tokens) >= 5:
        kept.append((sentence, content_tokens))

print("Sentences before the filter:", len(sentences))
print("Sentences after the filter :", len(kept), "(each has at least 5 non-stop-word tokens)")
print("Discarded                  :", len(sentences) - len(kept))

print("\n3 sample sentences:")
for sentence, content in kept[:3]:
    print("  sentence:", sentence)
    print("  tokens  :", content, "\n")

Sentences before the filter: 67
Sentences after the filter : 63 (each has at least 5 non-stop-word tokens)
Discarded                  : 4

3 sample sentences:
  sentence: introduction to natural language processing natural language processing usually shortened to nlp is the field of computer science that teaches machines to read understand and produce human language
  tokens  : ['introduction', 'natural', 'language', 'processing', 'natural', 'language', 'processing', 'usually', 'shortened', 'nlp', 'field', 'computer', 'science', 'teaches', 'machines', 'read', 'understand', 'produce', 'human', 'language'] 

  sentence: every day people write emails send messages post reviews and search the web and nearly all of this data is unstructured text
  tokens  : ['every', 'day', 'people', 'write', 'emails', 'send', 'messages', 'post', 'reviews', 'search', 'web', 'nearly', 'data', 'unstructured', 'text'] 

  sentence: nlp gives us the tools to turn that text into something a program can count com

### Task 03 (d)

> Write all remaining sentences to a file named nlp_ready_output.txt, one sentence per line. Re-read the file and print: (i) total lines written, (ii) the longest sentence by character length, and (iii) the shortest sentence by character length.

In [14]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 03 (d)
remaining = [sentence for sentence, _ in kept]            # the cleaned sentences that passed the filter

with open("nlp_ready_output.txt", "w", encoding="utf-8") as f:
    for sentence in remaining:
        f.write(sentence + "\n")

with open("nlp_ready_output.txt", encoding="utf-8") as f:
    lines = [line.rstrip("\n") for line in f]

longest = max(lines, key=len)
shortest = min(lines, key=len)
print("(i)   Total lines written:", len(lines))
print(f"(ii)  Longest sentence ({len(longest)} characters):\n      {longest}")
print(f"(iii) Shortest sentence ({len(shortest)} characters):\n      {shortest}")

(i)   Total lines written: 63
(ii)  Longest sentence (197 characters):
      introduction to natural language processing natural language processing usually shortened to nlp is the field of computer science that teaches machines to read understand and produce human language
(iii) Shortest sentence (36 characters):
      text preprocessing raw text is messy


---

## Task 04: DOCX Paragraph Classification and Statistical Analysis

**Topic:** Reading a Word document: paragraph styles and statistics

### Task 04 (a)

> Open any .docx file using python-docx. For each paragraph, store its text and style.name in a list of dictionaries. Print the total paragraph count and all unique style names found in the document.

In [15]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 04 (a)
document = docx.Document(DOCX_A)
paragraph_list = [{"text": p.text, "style": p.style.name} for p in document.paragraphs]

print("File:", DOCX_A)
print("Total paragraph count:", len(paragraph_list))
print("Unique style names   :", sorted({p["style"] for p in paragraph_list}))
print("\nFirst 6 entries:")
for entry in paragraph_list[:6]:
    print("  ", entry["style"].ljust(12), "|", entry["text"][:60])

File: data/report_a.docx
Total paragraph count: 29
Unique style names   : ['Body Text', 'Heading 1', 'Heading 2', 'List Bullet', 'Normal', 'Quote', 'Subtitle', 'Title']

First 6 entries:
   Title        | Natural Language Processing Lab Report
   Subtitle     | Prepared for the NLP laboratory course
   Normal       | This report summarises the topics covered in the laboratory 
   Normal       | Each session combines a short explanation with practical exe
   Heading 1    | Course Schedule
   Normal       | 


### Task 04 (b)

> Classify all paragraphs into exactly three groups: (i) Headings - style name starts with "Heading", (ii) Body - style is "Normal" or contains "Body", (iii) Other/Empty - everything else including blank paragraphs. Print the count of each group.

In [16]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 04 (b)
headings, body, other = [], [], []
for entry in paragraph_list:
    if not entry["text"].strip():
        other.append(entry)                                       # blank paragraphs always go to Other/Empty
    elif entry["style"].startswith("Heading"):
        headings.append(entry)
    elif entry["style"] == "Normal" or "Body" in entry["style"]:
        body.append(entry)
    else:
        other.append(entry)                                       # Title, Subtitle, List Bullet, Quote ...

print(f"Headings     : {len(headings)}")
print(f"Body         : {len(body)}")
print(f"Other/Empty  : {len(other)}")
print("Total        :", len(headings) + len(body) + len(other), "(equals", len(paragraph_list), "paragraphs)")
print("\nOther/Empty styles:", dict(collections.Counter(e["style"] + (" (blank)" if not e["text"].strip() else "") for e in other)))

Headings     : 7
Body         : 11
Other/Empty  : 11
Total        : 29 (equals 29 paragraphs)

Other/Empty styles: {'Title': 1, 'Subtitle': 1, 'Normal (blank)': 4, 'List Bullet': 4, 'Quote': 1}


### Task 04 (c)

> For every Body paragraph with more than 5 words, compute its word count and character count excluding spaces. Print: (i) the paragraph with the highest word count, (ii) the paragraph with the lowest word count above 5 words, and (iii) the mean word count across all qualifying body paragraphs rounded to 2 decimal places.

In [17]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 04 (c)
qualifying = []
for entry in body:
    words = len(entry["text"].split())
    if words > 5:
        qualifying.append({"text": entry["text"], "words": words, "chars": len(entry["text"].replace(" ", ""))})

print("Body paragraphs with more than 5 words:", len(qualifying))
highest = max(qualifying, key=lambda e: e["words"])
lowest = min(qualifying, key=lambda e: e["words"])
mean_words = sum(e["words"] for e in qualifying) / len(qualifying)

print(f"\n(i)   Highest word count ({highest['words']} words, {highest['chars']} characters without spaces):")
print("      ", highest["text"])
print(f"\n(ii)  Lowest word count above 5 ({lowest['words']} words, {lowest['chars']} characters without spaces):")
print("      ", lowest["text"])
print(f"\n(iii) Mean word count of the qualifying paragraphs: {mean_words:.2f}")

Body paragraphs with more than 5 words: 11

(i)   Highest word count (29 words, 121 characters without spaces):
       Stop words are very common words such as the, is and of. Removing them reduces the size of the text, but it can also remove words that carry meaning.

(ii)  Lowest word count above 5 (15 words, 93 characters without spaces):
       Each session combines a short explanation with practical exercises that students solve in Python notebooks.

(iii) Mean word count of the qualifying paragraphs: 19.45


### Task 04 (d)

> Build a nested dictionary structured as { style_name: { "count": N, "texts": [ list ] } } for all styles found. Sort the outer dictionary by count in descending order. If a style has more than 2 paragraphs, show only the first 2 texts. Pretty-print the result.

In [18]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 04 (d)
by_style = {}
for entry in paragraph_list:
    slot = by_style.setdefault(entry["style"], {"count": 0, "texts": []})
    slot["count"] += 1                                   # every paragraph of the style is counted, blank ones too
    if entry["text"].strip():
        slot["texts"].append(entry["text"])              # but only real text is listed

# sort the outer dictionary by count, largest first
sorted_styles = dict(sorted(by_style.items(), key=lambda kv: kv[1]["count"], reverse=True))

# a style with more than 2 paragraphs shows only its first 2 texts
display = {name: {"count": info["count"], "texts": info["texts"][:2] if info["count"] > 2 else info["texts"]}
           for name, info in sorted_styles.items()}

pprint.pprint(display, width=110, sort_dicts=False)

{'Normal': {'count': 13,
            'texts': ['This report summarises the topics covered in the laboratory sessions and the tools '
                      'that were used in each week.',
                      'Each session combines a short explanation with practical exercises that students '
                      'solve in Python notebooks.']},
 'Heading 1': {'count': 6, 'texts': ['Course Schedule', 'Introduction']},
 'List Bullet': {'count': 4,
                 'texts': ['NLTK for tokenizers, stop words and corpora',
                           'spaCy for fast pipelines and linguistic attributes']},
 'Body Text': {'count': 2,
               'texts': ['Each topic is first explained in a few sentences and then practised with a short '
                         'program.',
                         'A tokenizer must decide what to do with contractions, hyphens and abbreviations, '
                         'and different libraries make different choices.']},
 'Title': {'count': 1, 'texts': [

---

## Task 05: DOCX Table Extraction and Pandas Integration

**Topic:** Reading tables from a Word document and loading them into pandas

### Task 05 (a)

> Open a .docx file that contains at least one table. Iterate through doc.tables and extract each table as a list of lists: [[row1col1, row1col2, ...], [row2col1, ...]]. Print the total number of tables found and the dimensions (rows x columns) of each.

In [19]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 05 (a)
doc5 = docx.Document(DOCX_A)
all_tables = []
for table in doc5.tables:
    all_tables.append([[cell.text.strip() for cell in row.cells] for row in table.rows])

print("Total number of tables found:", len(all_tables))
for i, t in enumerate(all_tables, 1):
    print(f"   table {i}: {len(t)} rows x {len(t[0])} columns")

print("\nTable 1 as a list of lists:")
pprint.pprint(all_tables[0], width=100)

Total number of tables found: 2
   table 1: 7 rows x 4 columns
   table 2: 3 rows x 2 columns

Table 1 as a list of lists:
[['Week', 'Topic', 'Library', 'Marks'],
 ['1', 'Python for NLP', 'Python', '5'],
 ['2', 'Regular expressions', 're', '5'],
 ['3', 'Tokenization', 'NLTK, spaCy', ''],
 ['4', 'Text preprocessing', 'NLTK', '10'],
 ['5', 'Language models', '', '10'],
 ['6', 'Evaluation', 'scikit-learn', '']]


### Task 05 (b)

> Take the first table's data, treat the first row as the header and the remaining rows as data, and create a pandas DataFrame. Print the full DataFrame. Then count and print the number of empty or None cells per column.

In [20]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 05 (b)
first_table = all_tables[0]
df = pd.DataFrame(first_table[1:], columns=first_table[0])

print("Full DataFrame:")
print(df.to_string(index=False))

# a cell counts as empty when it is None/NaN or an empty string
empty_per_column = df.isna().sum() + (df == "").sum()
print("\nEmpty or None cells per column:")
print(empty_per_column.to_string())
print("Total empty cells:", int(empty_per_column.sum()))

Full DataFrame:
Week               Topic      Library Marks
   1      Python for NLP       Python     5
   2 Regular expressions           re     5
   3        Tokenization  NLTK, spaCy      
   4  Text preprocessing         NLTK    10
   5     Language models                 10
   6          Evaluation scikit-learn      

Empty or None cells per column:
Week       0
Topic      0
Library    1
Marks      2
Total empty cells: 3


### Task 05 (c)

> Extract all paragraphs that appear before the first table in the document by iterating through doc.element.body children and stopping once a w:tbl element is encountered. Store these paragraphs in a list and print their count.

In [21]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 05 (c)
pre_table_paragraphs = []
for child in doc5.element.body.iterchildren():
    if child.tag.endswith("}tbl"):                     # w:tbl = the first table: stop here
        break
    if child.tag.endswith("}p"):                       # w:p = a paragraph
        pre_table_paragraphs.append(Paragraph(child, doc5).text)

print("Paragraphs before the first table:", len(pre_table_paragraphs))
for i, text in enumerate(pre_table_paragraphs, 1):
    print(f"   {i}. {text}")

Paragraphs before the first table: 5
   1. Natural Language Processing Lab Report
   2. Prepared for the NLP laboratory course
   3. This report summarises the topics covered in the laboratory sessions and the tools that were used in each week.
   4. Each session combines a short explanation with practical exercises that students solve in Python notebooks.
   5. Course Schedule


### Task 05 (d)

> Write both the pre-table paragraphs and the first table's data into a single file named docx_extracted.txt. The file must have clearly labeled sections: a --- PRE-TABLE PARAGRAPHS --- header followed by the paragraphs, then a --- TABLE DATA --- header followed by rows formatted as pipe-separated values, e.g., col1 | col2 | col3.

In [22]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 05 (d)
with open("docx_extracted.txt", "w", encoding="utf-8") as f:
    f.write("--- PRE-TABLE PARAGRAPHS ---\n")
    for text in pre_table_paragraphs:
        f.write(text + "\n")
    f.write("\n--- TABLE DATA ---\n")
    for row in first_table:
        f.write(" | ".join(row) + "\n")

print("Contents of docx_extracted.txt:\n")
print(open("docx_extracted.txt", encoding="utf-8").read())

Contents of docx_extracted.txt:

--- PRE-TABLE PARAGRAPHS ---
Natural Language Processing Lab Report
Prepared for the NLP laboratory course
This report summarises the topics covered in the laboratory sessions and the tools that were used in each week.
Each session combines a short explanation with practical exercises that students solve in Python notebooks.
Course Schedule

--- TABLE DATA ---
Week | Topic | Library | Marks
1 | Python for NLP | Python | 5
2 | Regular expressions | re | 5
3 | Tokenization | NLTK, spaCy | 
4 | Text preprocessing | NLTK | 10
5 | Language models |  | 10
6 | Evaluation | scikit-learn | 



---

## Task 06: Multi-Document Comparison, Deduplication, and Frequency Report

**Topic:** Comparing two Word documents

### Task 06 (a)

> Open two different .docx files (Document A and Document B). From each, extract all paragraphs whose style name starts with "Heading" into two separate lists. Using Python set operations, print: (i) headings common to both documents, (ii) headings only in Document A, and (iii) headings only in Document B.

In [23]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 06 (a)
doc_a = docx.Document(DOCX_A)
doc_b = docx.Document(DOCX_B)

headings_a = [p.text.strip() for p in doc_a.paragraphs if p.style.name.startswith("Heading")]
headings_b = [p.text.strip() for p in doc_b.paragraphs if p.style.name.startswith("Heading")]

print("Headings in Document A:", headings_a)
print("Headings in Document B:", headings_b)

set_a, set_b = set(headings_a), set(headings_b)
print("\n(i)   Common to both  :", sorted(set_a & set_b))
print("(ii)  Only in A       :", sorted(set_a - set_b))
print("(iii) Only in B       :", sorted(set_b - set_a))

Headings in Document A: ['Course Schedule', 'Introduction', 'Tokenization', 'Libraries Used', 'Preprocessing', 'Evaluation', 'Conclusion']
Headings in Document B: ['Introduction', 'Tokenization', 'Stemming and Lemmatization', 'Conclusion']

(i)   Common to both  : ['Conclusion', 'Introduction', 'Tokenization']
(ii)  Only in A       : ['Course Schedule', 'Evaluation', 'Libraries Used', 'Preprocessing']
(iii) Only in B       : ['Stemming and Lemmatization']


### Task 06 (b)

> From each document, extract all "Normal" style paragraphs. Merge both lists. Remove exact duplicate paragraphs by comparing stripped, lowercased text. Print the total count before and after deduplication.

In [24]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 06 (b)
def normal_paragraphs(document):
    """Text of every 'Normal' paragraph. Blank paragraphs hold no content, so they are skipped."""
    return [p.text for p in document.paragraphs if p.style.name == "Normal" and p.text.strip()]


merged_paragraphs = normal_paragraphs(doc_a) + normal_paragraphs(doc_b)

seen = set()
unique_paragraphs = []
for text in merged_paragraphs:
    key = text.strip().lower()                  # compare the stripped, lowercased text
    if key not in seen:
        seen.add(key)
        unique_paragraphs.append(text.strip())

print("Normal paragraphs in A:", len(normal_paragraphs(doc_a)))
print("Normal paragraphs in B:", len(normal_paragraphs(doc_b)))
print("Total before deduplication:", len(merged_paragraphs))
print("Total after deduplication :", len(unique_paragraphs))
print("Duplicates removed        :", len(merged_paragraphs) - len(unique_paragraphs))

Normal paragraphs in A: 9
Normal paragraphs in B: 7
Total before deduplication: 16
Total after deduplication : 13
Duplicates removed        : 3


### Task 06 (c)

> Tokenize every paragraph in the deduplicated list into words (lowercase, punctuation removed via re.sub()). Build a combined word frequency dictionary using collections.Counter. Remove at least 15 stopwords. Display the top 20 most frequent words.

In [25]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 06 (c)
all_tokens = []
for text in unique_paragraphs:
    cleaned_text = re.sub(r"[^\w\s]", " ", text.lower())          # remove punctuation
    all_tokens.extend(cleaned_text.split())

counts = collections.Counter(t for t in all_tokens if t not in STOP_WORDS)
print("Stop words removed:", len(STOP_WORDS), "(more than the 15 required)")
print("Tokens before / after stop-word removal:", len(all_tokens), "/", sum(counts.values()))

print("\nTop 20 most frequent words:")
for rank, (word, n) in enumerate(sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))[:20], 1):
    print(f"   {rank:>2}. {word:<14} {n}")

Stop words removed: 198 (more than the 15 required)
Tokens before / after stop-word removal: 260 / 150

Top 20 most frequent words:
    1. text           7
    2. used           4
    3. words          4
    4. students       3
    5. change         2
    6. find           2
    7. laboratory     2
    8. language       2
    9. model          2
   10. order          2
   11. practical      2
   12. processing     2
   13. raw            2
   14. sentence       2
   15. sessions       2
   16. short          2
   17. stemming       2
   18. step           2
   19. stop           2
   20. word           2


### Task 06 (d)

> Compute and print a summary containing: (i) total headings across both documents, (ii) total unique body paragraphs after deduplication, (iii) total unique word types (vocabulary size), (iv) average paragraph length in words rounded to 2 decimal places, and (v) the single longest paragraph by word count from the combined set.

In [26]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 06 (d)
lengths = [len(p.split()) for p in unique_paragraphs]
longest_paragraph = max(unique_paragraphs, key=lambda p: len(p.split()))

print("SUMMARY")
print("-" * 60)
print("(i)   Total headings across both documents :", len(headings_a) + len(headings_b))
print("(ii)  Unique body paragraphs after dedup   :", len(unique_paragraphs))
print("(iii) Vocabulary size (unique word types)  :", len(set(all_tokens)))
print(f"(iv)  Average paragraph length             : {sum(lengths) / len(lengths):.2f} words")
print(f"(v)   Longest paragraph ({len(longest_paragraph.split())} words):")
print("      ", longest_paragraph)

SUMMARY
------------------------------------------------------------
(i)   Total headings across both documents : 11
(ii)  Unique body paragraphs after dedup   : 13
(iii) Vocabulary size (unique word types)  : 154
(iv)  Average paragraph length             : 20.00 words
(v)   Longest paragraph (29 words):
       Stop words are very common words such as the, is and of. Removing them reduces the size of the text, but it can also remove words that carry meaning.


---

## Task 07: Multi-Endpoint API Data Joining and Analysis

**Topic:** Joining two API endpoints

### Task 07 (a)

> Send GET requests to both endpoints below. For each, verify the HTTP status code is exactly 200; if not, raise a ConnectionError with the actual status code in the message. Parse both responses as JSON lists.

In [27]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 07 (a)
BASE = "https://jsonplaceholder.typicode.com"


def fetch_json(url):
    """GET a URL, insist on status 200, and return the parsed JSON."""
    response = requests.get(url, timeout=15)
    if response.status_code != 200:
        raise ConnectionError(f"Request to {url} failed with status code {response.status_code}")
    return response.json()


users = fetch_json(f"{BASE}/users")
posts = fetch_json(f"{BASE}/posts")

print("users:", type(users).__name__, "with", len(users), "records")
print("posts:", type(posts).__name__, "with", len(posts), "records")

# The error path: a URL that does not exist gives a 404, which must raise ConnectionError
try:
    fetch_json(f"{BASE}/not-a-real-endpoint")
except ConnectionError as error:
    print("\nCaught ConnectionError ->", error)

users: list with 10 records
posts: list with 100 records



Caught ConnectionError -> Request to https://jsonplaceholder.typicode.com/not-a-real-endpoint failed with status code 404


### Task 07 (b)

> Join the two datasets using id from users and userId from posts. Build a dictionary structured as: { user_name: { "email": "...", "post_count": N, "titles": [ list of post titles ] } }. Print the result for the first 3 users only.

In [28]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 07 (b)
joined = {}
for user in users:
    user_posts = [p for p in posts if p["userId"] == user["id"]]          # join on id == userId
    joined[user["name"]] = {
        "email": user["email"],
        "post_count": len(user_posts),
        "titles": [p["title"] for p in user_posts],
    }

print("Joined dictionary has", len(joined), "users. First 3:\n")
for name in list(joined)[:3]:
    info = joined[name]
    print(name)
    print("   email     :", info["email"])
    print("   post_count:", info["post_count"])
    print("   titles    :")
    for title in info["titles"][:3]:
        print("      -", title)
    print(f"      ... ({info['post_count'] - 3} more titles)\n")

Joined dictionary has 10 users. First 3:

Leanne Graham
   email     : Sincere@april.biz
   post_count: 10
   titles    :
      - sunt aut facere repellat provident occaecati excepturi optio reprehenderit
      - qui est esse
      - ea molestias quasi exercitationem repellat qui ipsa sit aut
      ... (7 more titles)

Ervin Howell
   email     : Shanna@melissa.tv
   post_count: 10
   titles    :
      - et ea vero quia laudantium autem
      - in quibusdam tempore odit est dolorem
      - dolorum ut in voluptas mollitia et saepe quo animi
      ... (7 more titles)

Clementine Bauch
   email     : Nathan@yesenia.net
   post_count: 10
   titles    :
      - asperiores ea ipsam voluptatibus modi minima quia sint
      - dolor sint quo a velit explicabo quia nam
      - maxime id vitae nihil numquam
      ... (7 more titles)



### Task 07 (c)

> From the joined dictionary, find the user who has written the most posts. Print their: (i) full name, (ii) email address, (iii) company name, and (iv) all post titles numbered.

In [29]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 07 (c)
most_posts = max(info["post_count"] for info in joined.values())
leaders = [name for name, info in joined.items() if info["post_count"] == most_posts]
top_name = leaders[0]                                      # on a tie the first user is used
top_user = next(u for u in users if u["name"] == top_name)

if len(leaders) > 1:
    print(f"Note: {len(leaders)} users tie with {most_posts} posts each, so the first of them is shown.\n")

print("(i)   Full name    :", top_name)
print("(ii)  Email        :", joined[top_name]["email"])
print("(iii) Company name :", top_user["company"]["name"])
print("(iv)  Post titles  :")
for i, title in enumerate(joined[top_name]["titles"], 1):
    print(f"      {i:>2}. {title}")

Note: 10 users tie with 10 posts each, so the first of them is shown.

(i)   Full name    : Leanne Graham
(ii)  Email        : Sincere@april.biz
(iii) Company name : Romaguera-Crona
(iv)  Post titles  :
       1. sunt aut facere repellat provident occaecati excepturi optio reprehenderit
       2. qui est esse
       3. ea molestias quasi exercitationem repellat qui ipsa sit aut
       4. eum et est occaecati
       5. nesciunt quas odio
       6. dolorem eum magni eos aperiam quia
       7. magnam facilis autem
       8. dolorem dolore est ipsam
       9. nesciunt iure omnis dolorem tempora et accusantium
      10. optio molestias id quia eum


### Task 07 (d)

> Filter users where either of the following is true: (i) company.bs field contains the word "synergies" (case-insensitive), or (ii) address.city has more than 8 characters. Print the name, city, and post count for all matching users.

In [30]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 07 (d)
print("company.bs values:")
for u in users:
    print(f"   {u['name']:<26} {u['company']['bs']}")

matches = []
for u in users:
    has_synergies = "synergies" in u["company"]["bs"].lower()
    long_city = len(u["address"]["city"]) > 8
    if has_synergies or long_city:
        matches.append((u["name"], u["address"]["city"], joined[u["name"]]["post_count"], has_synergies, long_city))

print(f"\nUsers matching (synergies in company.bs OR city longer than 8 characters): {len(matches)}")
print(f"{'Name':<26}{'City':<16}{'Posts':>6}   reason")
for name, city, count, syn, long_ in matches:
    reason = " + ".join(r for r, ok in [("synergies", syn), ("city > 8 chars", long_)] if ok)
    print(f"{name:<26}{city:<16}{count:>6}   {reason}")
if not any(m[3] for m in matches):
    print("\n(No company.bs contains the word 'synergies' in this data; all matches come from the long city names.)")

company.bs values:
   Leanne Graham              harness real-time e-markets
   Ervin Howell               synergize scalable supply-chains
   Clementine Bauch           e-enable strategic applications
   Patricia Lebsack           transition cutting-edge web services
   Chelsey Dietrich           revolutionize end-to-end systems
   Mrs. Dennis Schulist       e-enable innovative applications
   Kurtis Weissnat            generate enterprise e-tailers
   Nicholas Runolfsdottir V   e-enable extensible e-tailers
   Glenna Reichert            aggregate real-time technologies
   Clementina DuBuque         target end-to-end models

Users matching (synergies in company.bs OR city longer than 8 characters): 10
Name                      City             Posts   reason
Leanne Graham             Gwenborough         10   city > 8 chars
Ervin Howell              Wisokyburgh         10   city > 8 chars
Clementine Bauch          McKenziehaven       10   city > 8 chars
Patricia Lebsack          South 

---

## Task 08: To-Do Completion Rate Analysis Across Users

**Topic:** Measuring how many to-dos each user completed

### Task 08 (a)

> Fetch data from the endpoint below and parse it into a Python list. Validate that: (i) the response status is 200, (ii) the parsed result is a list, and (iii) the list contains at least one record. If any validation fails, print a meaningful error and exit. Print the total number of todo records retrieved.

In [31]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 08 (a)
response = requests.get(f"{BASE}/todos", timeout=15)

if response.status_code != 200:                                    # (i)
    print(f"Error: the server answered with status {response.status_code} instead of 200.")
    sys.exit(1)

todos = response.json()
if not isinstance(todos, list):                                    # (ii)
    print(f"Error: expected a list but received a {type(todos).__name__}.")
    sys.exit(1)
if len(todos) < 1:                                                 # (iii)
    print("Error: the list of todos is empty.")
    sys.exit(1)

print("All three validations passed (status 200, result is a list, at least one record).")
print("Total number of todo records retrieved:", len(todos))
print("First record:", todos[0])

All three validations passed (status 200, result is a list, at least one record).
Total number of todo records retrieved: 200
First record: {'userId': 1, 'id': 1, 'title': 'delectus aut autem', 'completed': False}


### Task 08 (b)

> Group todos by userId. For each user, compute and store: { userId: { "total": N, "completed": N, "incomplete": N } }. Print the result for all 10 users. For each user, verify that completed + incomplete == total and print True or False accordingly.

In [32]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 08 (b)
stats = {}
for todo in todos:
    s = stats.setdefault(todo["userId"], {"total": 0, "completed": 0, "incomplete": 0})
    s["total"] += 1
    if todo["completed"]:
        s["completed"] += 1
    else:
        s["incomplete"] += 1

print(f"{'userId':>6}{'total':>7}{'completed':>11}{'incomplete':>12}   completed + incomplete == total")
for user_id in sorted(stats):
    s = stats[user_id]
    print(f"{user_id:>6}{s['total']:>7}{s['completed']:>11}{s['incomplete']:>12}   {s['completed'] + s['incomplete'] == s['total']}")
print("\nUsers:", len(stats))

userId  total  completed  incomplete   completed + incomplete == total
     1     20         11           9   True
     2     20          8          12   True
     3     20          7          13   True
     4     20          6          14   True
     5     20         12           8   True
     6     20          6          14   True
     7     20          9          11   True
     8     20         11           9   True
     9     20          8          12   True
    10     20         12           8   True

Users: 10


### Task 08 (c)

> Compute the completion rate for each user as completed / total x 100 rounded to 1 decimal place. Print: (i) the userId with the highest completion rate and its value, (ii) the userId with the lowest completion rate and its value, and (iii) the overall average completion rate across all users.

In [33]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 08 (c)
rates = {uid: round(s["completed"] / s["total"] * 100, 1) for uid, s in stats.items()}

best_user = max(rates, key=rates.get)
worst_user = min(rates, key=rates.get)
average_rate = sum(rates.values()) / len(rates)

print("Completion rate of every user (%):", dict(sorted(rates.items())))
print(f"\n(i)   Highest completion rate: userId {best_user} with {rates[best_user]}%")
print(f"(ii)  Lowest completion rate : userId {worst_user} with {rates[worst_user]}%")
print(f"(iii) Overall average completion rate across all users: {average_rate:.1f}%")

Completion rate of every user (%): {1: 55.0, 2: 40.0, 3: 35.0, 4: 30.0, 5: 60.0, 6: 30.0, 7: 45.0, 8: 55.0, 9: 40.0, 10: 60.0}

(i)   Highest completion rate: userId 5 with 60.0%
(ii)  Lowest completion rate : userId 4 with 30.0%
(iii) Overall average completion rate across all users: 45.0%


### Task 08 (d)

> Fetch https://jsonplaceholder.typicode.com/users and build a lookup dictionary { userId: user_name }. Print a formatted table with columns: Rank | Name | Total Tasks | Completed | Completion Rate %, sorted by completion rate in descending order. Use f-strings or str.ljust() to align columns neatly.

In [34]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 08 (d)
lookup = {u["id"]: u["name"] for u in fetch_json(f"{BASE}/users")}

ranking = sorted(stats, key=lambda uid: (-rates[uid], uid))              # best rate first

header = f"{'Rank':<6}{'Name':<26}{'Total Tasks':>12}{'Completed':>11}{'Completion Rate %':>20}"
print(header)
print("-" * len(header))
for rank, uid in enumerate(ranking, 1):
    s = stats[uid]
    print(f"{rank:<6}{lookup[uid]:<26}{s['total']:>12}{s['completed']:>11}{rates[uid]:>20.1f}")

Rank  Name                       Total Tasks  Completed   Completion Rate %
---------------------------------------------------------------------------
1     Chelsey Dietrich                    20         12                60.0
2     Clementina DuBuque                  20         12                60.0
3     Leanne Graham                       20         11                55.0
4     Nicholas Runolfsdottir V            20         11                55.0
5     Kurtis Weissnat                     20          9                45.0
6     Ervin Howell                        20          8                40.0
7     Glenna Reichert                     20          8                40.0
8     Clementine Bauch                    20          7                35.0
9     Patricia Lebsack                    20          6                30.0
10    Mrs. Dennis Schulist                20          6                30.0


---

## Task 09: Nested JSON Flattening, DataFrame Operations, and Comment Analysis

**Topic:** Flattening nested JSON, working with pandas, and counting comments per post

### Task 09 (a)

> Fetch data from https://jsonplaceholder.typicode.com/users. Each user contains nested objects (address with geo, and company). Write a function flatten_user(user: dict) -> dict that returns a single-level dictionary with exactly these keys: id, name, username, email, city, zipcode, lat, lng, company_name, company_catchphrase. Apply this function to all 10 users and print the resulting flattened list.

In [35]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 09 (a)
def flatten_user(user: dict) -> dict:
    """Turn the nested user record into a single-level dictionary with exactly 10 keys."""
    return {
        "id": user["id"],
        "name": user["name"],
        "username": user["username"],
        "email": user["email"],
        "city": user["address"]["city"],
        "zipcode": user["address"]["zipcode"],
        "lat": user["address"]["geo"]["lat"],
        "lng": user["address"]["geo"]["lng"],
        "company_name": user["company"]["name"],
        "company_catchphrase": user["company"]["catchPhrase"],
    }


users9 = fetch_json(f"{BASE}/users")
flat_users = [flatten_user(u) for u in users9]

print("Users flattened:", len(flat_users), "| keys per record:", len(flat_users[0]))
print("Keys:", list(flat_users[0]))
print()
for record in flat_users:
    print(record)

Users flattened: 10 | keys per record: 10
Keys: ['id', 'name', 'username', 'email', 'city', 'zipcode', 'lat', 'lng', 'company_name', 'company_catchphrase']

{'id': 1, 'name': 'Leanne Graham', 'username': 'Bret', 'email': 'Sincere@april.biz', 'city': 'Gwenborough', 'zipcode': '92998-3874', 'lat': '-37.3159', 'lng': '81.1496', 'company_name': 'Romaguera-Crona', 'company_catchphrase': 'Multi-layered client-server neural-net'}
{'id': 2, 'name': 'Ervin Howell', 'username': 'Antonette', 'email': 'Shanna@melissa.tv', 'city': 'Wisokyburgh', 'zipcode': '90566-7771', 'lat': '-43.9509', 'lng': '-34.4618', 'company_name': 'Deckow-Crist', 'company_catchphrase': 'Proactive didactic contingency'}
{'id': 3, 'name': 'Clementine Bauch', 'username': 'Samantha', 'email': 'Nathan@yesenia.net', 'city': 'McKenziehaven', 'zipcode': '59590-4157', 'lat': '-68.6102', 'lng': '-47.0653', 'company_name': 'Romaguera-Jacobson', 'company_catchphrase': 'Face to face bifurcated interface'}
{'id': 4, 'name': 'Patricia Le

### Task 09 (b)

> Create a pandas DataFrame from the flattened user list and display it in full. Then: (i) cast the lat and lng columns to float, (ii) add a new column distance_from_origin computed as sqrt(lat^2 + lng^2) using the math module applied row-by-row, and (iii) print the user with the greatest distance from origin.

In [36]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 09 (b)
users_df = pd.DataFrame(flat_users)
print("The DataFrame as created (lat and lng are still text):")
print(users_df[["id", "name", "city", "lat", "lng"]].to_string(index=False))
print("\ndtypes before:", users_df["lat"].dtype, users_df["lng"].dtype)

users_df["lat"] = users_df["lat"].astype(float)                     # (i)
users_df["lng"] = users_df["lng"].astype(float)
print("dtypes after :", users_df["lat"].dtype, users_df["lng"].dtype)

users_df["distance_from_origin"] = users_df.apply(                  # (ii) row by row with math.sqrt
    lambda row: math.sqrt(row["lat"] ** 2 + row["lng"] ** 2), axis=1)

print("\nDataFrame with the new column:")
print(users_df[["name", "lat", "lng", "distance_from_origin"]].round(4).to_string(index=False))

farthest = users_df.loc[users_df["distance_from_origin"].idxmax()]  # (iii)
print(f"\nGreatest distance from the origin: {farthest['name']} "
      f"(lat {farthest['lat']}, lng {farthest['lng']}) -> {farthest['distance_from_origin']:.4f}")

The DataFrame as created (lat and lng are still text):
 id                     name           city      lat       lng
  1            Leanne Graham    Gwenborough -37.3159   81.1496
  2             Ervin Howell    Wisokyburgh -43.9509  -34.4618
  3         Clementine Bauch  McKenziehaven -68.6102  -47.0653
  4         Patricia Lebsack    South Elvis  29.4572 -164.2990
  5         Chelsey Dietrich     Roscoeview -31.8129   62.5342
  6     Mrs. Dennis Schulist  South Christy -71.4197   71.7478
  7          Kurtis Weissnat      Howemouth  24.8918   21.8984
  8 Nicholas Runolfsdottir V      Aliyaview -14.3990 -120.7677
  9          Glenna Reichert Bartholomebury  24.6463 -168.8889
 10       Clementina DuBuque    Lebsackbury -38.2386   57.2232

dtypes before: str str
dtypes after : float64 float64

DataFrame with the new column:
                    name      lat       lng  distance_from_origin
           Leanne Graham -37.3159   81.1496               89.3182
            Ervin Howell -43.9509

### Task 09 (c)

> Using the lat column, classify users into two groups: (i) Northern Hemisphere where lat > 0, and (ii) Southern Hemisphere where lat < 0. Print the names of users in each group. Then determine which hemisphere has the higher average distance_from_origin.

In [37]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 09 (c)
north = users_df[users_df["lat"] > 0]
south = users_df[users_df["lat"] < 0]

print(f"Northern Hemisphere (lat > 0): {len(north)} users")
for name in north["name"]:
    print("   -", name)
print(f"\nSouthern Hemisphere (lat < 0): {len(south)} users")
for name in south["name"]:
    print("   -", name)

avg_north = north["distance_from_origin"].mean()
avg_south = south["distance_from_origin"].mean()
print(f"\nAverage distance_from_origin, Northern: {avg_north:.4f}")
print(f"Average distance_from_origin, Southern: {avg_south:.4f}")
print("Higher average:", "Northern Hemisphere" if avg_north > avg_south else "Southern Hemisphere")

Northern Hemisphere (lat > 0): 3 users
   - Patricia Lebsack
   - Kurtis Weissnat
   - Glenna Reichert

Southern Hemisphere (lat < 0): 7 users
   - Leanne Graham
   - Ervin Howell
   - Clementine Bauch
   - Chelsey Dietrich
   - Mrs. Dennis Schulist
   - Nicholas Runolfsdottir V
   - Clementina DuBuque

Average distance_from_origin, Northern: 123.5833
Average distance_from_origin, Southern: 84.3162
Higher average: Northern Hemisphere


### Task 09 (d)

> Fetch both endpoints below. For each post, count its number of comments by matching postId in comments against id in posts. Build a sorted list of tuples: (post_id, post_title[:40], comment_count) in descending order by comment count. Print the top 5 most-commented and bottom 5 least-commented posts.

In [38]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 09 (d)
posts9 = fetch_json(f"{BASE}/posts")
comments9 = fetch_json(f"{BASE}/comments")
print("posts:", len(posts9), "| comments:", len(comments9))

comment_counts = collections.Counter(c["postId"] for c in comments9)       # postId -> number of comments

ranked = [(p["id"], p["title"][:40], comment_counts.get(p["id"], 0)) for p in posts9]
ranked.sort(key=lambda t: (-t[2], t[0]))                                    # most comments first, then by id

print("\nTop 5 most-commented posts:")
for t in ranked[:5]:
    print("  ", t)
print("\nBottom 5 least-commented posts:")
for t in ranked[-5:]:
    print("  ", t)

distinct_counts = sorted(set(t[2] for t in ranked))
print("\nDifferent comment counts found among the posts:", distinct_counts)
if len(distinct_counts) == 1:
    print(f"Every post has exactly {distinct_counts[0]} comments in this data, so the top and the bottom lists are")
    print("only ordered by post id; there is no 'most' or 'least' commented post.")

posts: 100 | comments: 500

Top 5 most-commented posts:
   (1, 'sunt aut facere repellat provident occae', 5)
   (2, 'qui est esse', 5)
   (3, 'ea molestias quasi exercitationem repell', 5)
   (4, 'eum et est occaecati', 5)
   (5, 'nesciunt quas odio', 5)

Bottom 5 least-commented posts:
   (96, 'quaerat velit veniam amet cupiditate aut', 5)
   (97, 'quas fugiat ut perspiciatis vero provide', 5)
   (98, 'laboriosam dolor voluptates', 5)
   (99, 'temporibus sit alias delectus eligendi p', 5)
   (100, 'at nam consequatur ea labore ea harum', 5)

Different comment counts found among the posts: [5]
Every post has exactly 5 comments in this data, so the top and the bottom lists are
only ordered by post id; there is no 'most' or 'least' commented post.


---

## End of Lab 02

All 9 tasks (36 parts) completed by **Zain Ul Wahaj** - **FA23-BAI-058**.